In [1]:
import sys
print(sys.executable)

/home/ranmaoyin/miniconda3/envs/post-train/bin/python


In [6]:
import sys

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(device))

model_id = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device).eval()

print("模型设备:", next(model.parameters()).device)
print("模型词表大小:", model.config.vocab_size)

GPU: NVIDIA GeForce RTX 4070 Laptop GPU


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 272/272 [00:00<00:00, 1380.24it/s]


模型设备: cuda:0
模型词表大小: 49152


In [3]:
prompts = ["2 + 3 =", "2 + 3 = "]
encoded = {}

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    encoded[prompt] = inputs

    token_ids = inputs["input_ids"][0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(token_ids)

    print(f"输入文本: {prompt!r}")
    print(f"token 数量: {len(token_ids)}")
    print("tokens:", tokens)
    print("input_ids:", token_ids)
    print()

输入文本: '2 + 3 ='
token 数量: 5
tokens: ['2', 'Ġ+', 'Ġ', '3', 'Ġ=']
input_ids: [34, 1232, 216, 35, 446]

输入文本: '2 + 3 = '
token 数量: 6
tokens: ['2', 'Ġ+', 'Ġ', '3', 'Ġ=', 'Ġ']
input_ids: [34, 1232, 216, 35, 446, 216]



In [4]:
last_logits = {}

with torch.inference_mode():
    for prompt, inputs in encoded.items():
        inputs = inputs.to(device)
        logits = model(**inputs, use_cache=False).logits

        print(f"输入文本: {prompt!r}")
        print(f"input_ids.shape: {tuple(inputs['input_ids'].shape)}")
        print(f"logits.shape: {tuple(logits.shape)}")
        print(f"logits.device: {logits.device}")
        print()

        last_logits[prompt] = logits[0, -1, :].float().cpu()

输入文本: '2 + 3 ='
input_ids.shape: (1, 5)
logits.shape: (1, 5, 49152)
logits.device: cuda:0

输入文本: '2 + 3 = '
input_ids.shape: (1, 6)
logits.shape: (1, 6, 49152)
logits.device: cuda:0



In [7]:
for prompt, scores in last_logits.items():
    next_probs = scores.softmax(dim=-1)
    top_ids = next_probs.topk(5).indices.tolist()
    bottom_ids = scores.topk(5, largest=False).indices.tolist()

    print(f"输入文本: {prompt!r}")

    for title, token_ids in [
        ("概率最高的 5 个候选（从高到低）", top_ids),
        ("logit 最低的 5 个候选（从低到高）", bottom_ids),
    ]:
        print(title)
        print(f"{'序号':<6}{'Token ID':<12}{'候选文本':<14}{'Logit':>10}{'概率':>14}")

        for rank, token_id in enumerate(token_ids, start=1):
            text = tokenizer.decode(
                [token_id],
                clean_up_tokenization_spaces=False,
            )
            logit = scores[token_id].item()
            prob = next_probs[token_id].item()
            print(f"{rank:<6}{token_id:<12}{text!r:<14}{logit:>10.4f}{prob:>14.4e}")

        print()

输入文本: '2 + 3 ='
概率最高的 5 个候选（从高到低）
序号    Token ID    候选文本               Logit            概率
1     216         ' '              21.8750    8.5569e-01
2     365         ' ('             17.6250    1.2206e-02
3     731         ' -'             17.5000    1.0772e-02
4     198         '\n'             17.2500    8.3889e-03
5     1885        ' $'             16.8750    5.7656e-03

logit 最低的 5 个候选（从低到高）
序号    Token ID    候选文本               Logit            概率
1     43476       'MetaInfo'      -11.5625    2.5739e-15
2     47595       'HETATM'        -10.9375    4.8087e-15
3     48518       'Intf'          -10.8125    5.4490e-15
4     4813        'ortunately'    -10.3125    8.9839e-15
5     27580       'geoId'         -10.0000    1.2280e-14

输入文本: '2 + 3 = '
概率最高的 5 个候选（从高到低）
序号    Token ID    候选文本               Logit            概率
1     33          '1'              34.0000    2.1642e-01
2     38          '6'              33.5000    1.3126e-01
3     36          '4'              33.5000    1.3126